# Chapter 5: Cell Complexes

**Source Span:** John M. Lee, *Introduction to Topological Manifolds*, Second Edition, Chapter 5, printed pp. 127-158, PDF pp. 145-176.

This notebook is a standalone computational lesson on cell complexes. The source chapter is used here only for orientation, terminology, and structure; the explanations, diagrams, checks, and examples are original.

## Chapter Goal

The goal is to make a CW complex feel like data that still remembers topology. A cell decomposition is not just a list of pieces. It is a list of open cells, characteristic maps from closed cells, attaching maps on boundaries, and a topology that lets local checks on cell closures control global continuity and closedness. By the end, you should be able to inspect a proposed cell structure and ask:

1. What are the cells and their dimensions?
2. How does each closed cell attach along its boundary?
3. Do the closure-finiteness condition (C) and coherent-topology condition (W) hold?
4. Which skeleton already contains a topological property such as connectedness?
5. When a 1-dimensional CW complex is a manifold, which graph pattern forces it to be a circle, line, compact interval, or half-line?
6. How do simplicial complexes encode a special, combinatorial kind of regular CW complex?

The chapter's main computational translation is: replace an infinite topological object by finite pieces of incidence, attachment, and closure data whenever possible, then validate the claims that must remain true in the actual topology.


## Translation Guide

A topologist says **open n-cell** for a piece homeomorphic to an open ball and **closed n-cell** for a piece homeomorphic to a closed ball. Computationally, we represent cells by a dimension, a label, and a closure relation. A characteristic map is represented by a parametrized map from a model closed cell. Its restriction to the boundary is the attaching map.

A **cell decomposition** is a partition into open cells plus characteristic maps whose boundaries land in lower-dimensional cells. For a finite complex, this often behaves exactly as intuition suggests. For an infinite complex, two additional tests matter. Condition (C), closure finiteness, says each cell closure touches only finitely many cells. Condition (W), coherence with closed cells, says a set is closed in the whole space precisely when its intersection with every closed cell is closed there. In code, (C) becomes a finite-incidence assertion, while (W) is modeled by counterexamples: sequences can look closed on every cell closure but still have an unaccounted global limit if the topology is not coherent.

A **skeleton** is the subspace made from cells of dimension at most n. The 0-skeleton is vertices, the 1-skeleton is a graph, and higher skeleta are produced by attaching higher cells. The proofs in this chapter repeatedly use skeletons as checkpoints: connectedness is already visible in the 1-skeleton, compactness in a CW complex means closed and contained in a finite subcomplex, and inductive construction attaches n-cells to the previous skeleton.

For **1-manifolds**, regular CW structures reduce the classification to graph logic. Every edge has two endpoint vertices. Without boundary, every vertex is incident to exactly two edges. A connected finite degree-two chain closes into a circle. A connected infinite degree-two chain is a line. With boundary, degree-one endpoints give the compact interval or half-line.

For **simplicial complexes**, the data are even more rigid: closed simplices, all faces, legal intersections, and local finiteness. This notebook uses a finite abstract complex and lets Gudhi verify the same combinatorial information that the CW viewpoint reads as cells and attachments.


## Library Routing

| Chapter concept | Representation in this notebook | Library route | Why this library fits |
| --- | --- | --- | --- |
| Closed cells from convex sets | Radial map from a unit disk to a convex ellipse | NumPy + Matplotlib | The concept is a 2D proof picture with numerical ray checks. |
| Attaching maps and skeletons | Bouquet graph plus a 2-cell boundary word | Plotly + SymPy | Plotly keeps the attaching picture inspectable; SymPy checks exact boundary algebra. |
| Closure finiteness and coherent topology | Two failure-mode diagrams and finite incidence diagnostics | Matplotlib + NetworkX | Static diagrams make the pathologies visible; NetworkX records proof dependencies. |
| 1-manifold classification | Degree patterns in connected regular graphs | NetworkX + Matplotlib | The classification is graph-theoretic once a regular CW structure is chosen. |
| Simplicial complexes | Abstract face data, boundary matrices, and Betti numbers | Gudhi + SymPy + Matplotlib | Simplicial topology is naturally combinatorial, and Gudhi validates the finite complex. |
| Applied lab | Editable diagnostics table for candidate cell structures | Pandas + JSON checks | Tables expose which hypothesis fails before a theorem is applied. |


## Visual Storyboard

The saved storyboard lives at `artifacts/chapter-05-cell-complexes/checks/visual-storyboard.json`. Each visual has an inspection target and an invariant.

1. **Radial closed-cell model:** inspect how each ray from an interior point reaches the boundary exactly once; validate sampled boundary equations and interior inequalities.
2. **Attaching-map quotient:** inspect a 1-skeleton and a 2-cell boundary word; validate Euler characteristic, cycle rank, and zero boundary for loop edges.
3. **CW condition failures:** inspect why a sequence can defeat (W) and why infinitely many boundary vertices defeat (C); validate the encoded failure flags.
4. **Proof dependency graph:** inspect which hypotheses feed compactness, connectedness, subcomplex, and construction theorems; validate the dependency graph is connected as a scaffold.
5. **1-manifold classifier:** inspect degree patterns for circle, line, interval, and half-line; validate degree and compactness signatures.
6. **Simplicial realization:** inspect a finite abstract complex as a triangulated disk; validate face closure, boundary-squared-zero, and Betti numbers.
7. **Applied diagnostic lab:** edit small cell-data cases and see which CW or manifold conclusion is licensed; validate JSON and CSV summaries.


In [ ]:
from pathlib import Path
import json
import math
import os
import sys

import gudhi
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sympy as sp
from IPython.display import Markdown, display


def locate_book_root(start: Path | None = None) -> Path:
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "source_map.json").exists() and (candidate / "utils").exists():
            return candidate
    raise RuntimeError("Could not locate Introduction-to-Topological-Manifolds root")


BOOK_ROOT = locate_book_root(Path.cwd())
NOTEBOOK_DIR = Path.cwd()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import (  # noqa: E402
    assert_artifacts,
    chapter_artifact_root,
    display_artifact,
    save_csv,
    save_json,
    save_matplotlib,
    save_plotly_html,
)
from utils.topology import (  # noqa: E402
    abelianization_vector,
    boundary_matrix,
    cycle_rank_for_graph,
    euler_characteristic,
    simplex_boundary_squared_zero,
)

UNIT_KEY = "chapter-05-cell-complexes"
ARTIFACT_ROOT = chapter_artifact_root(UNIT_KEY, BOOK_ROOT)
FIG_DIR = ARTIFACT_ROOT / "figures"
HTML_DIR = ARTIFACT_ROOT / "html"
CHECK_DIR = ARTIFACT_ROOT / "checks"
TABLE_DIR = ARTIFACT_ROOT / "tables"


def nb_rel(path: Path) -> Path:
    return Path(os.path.relpath(Path(path), NOTEBOOK_DIR))


visual_storyboard = [
    {
        "item": "radial_closed_cell_model",
        "concept": "compact convex closed cells are closed balls",
        "representation": "ellipse with rays from an interior point",
        "library": "numpy + matplotlib",
        "artifact": "figures/radial-cell-homeomorphism.png",
        "inspection_target": "each ray has one boundary hit and interior samples remain inside the cell",
        "validation": "sampled boundary equation equals 1 and interior inequalities are strict",
    },
    {
        "item": "attaching_map_quotient",
        "concept": "characteristic maps and attaching maps",
        "representation": "bouquet 1-skeleton with a 2-cell boundary word",
        "library": "plotly + sympy",
        "artifact": "html/attaching-map-skeletons.html",
        "inspection_target": "how the 2-cell boundary is read as an attaching word in the 1-skeleton",
        "validation": "Euler characteristic, cycle rank, abelianization, and boundary matrix checks",
    },
    {
        "item": "cw_condition_failures",
        "concept": "closure finiteness and coherent topology are independent requirements",
        "representation": "two original failure-mode diagrams",
        "library": "matplotlib",
        "artifact": "figures/cw-conditions-failure-modes.png",
        "inspection_target": "where global limit points or infinitely many incident cells appear",
        "validation": "JSON flags record C/W pass-fail behavior",
    },
    {
        "item": "proof_dependency_graph",
        "concept": "which hypotheses power the chapter theorems",
        "representation": "directed dependency graph",
        "library": "networkx + matplotlib",
        "artifact": "figures/cw-proof-dependency-graph.png",
        "inspection_target": "hypotheses feeding connectedness, compactness, subcomplex, and construction results",
        "validation": "dependency scaffold is weakly connected and has theorem targets",
    },
    {
        "item": "one_manifold_classifier",
        "concept": "regular 1-dimensional CW complexes classify connected 1-manifolds",
        "representation": "degree patterns for S^1, R, [0,1], and [0,infty)",
        "library": "networkx + matplotlib",
        "artifact": "figures/one-manifold-classifier.png",
        "inspection_target": "finite cycle, bi-infinite chain window, compact path, and ray window",
        "validation": "degree signatures and compactness flags match the four models",
    },
    {
        "item": "simplicial_realization",
        "concept": "abstract simplicial complexes and geometric realizations",
        "representation": "triangulated square disk from two 2-simplices",
        "library": "gudhi + sympy + matplotlib",
        "artifact": "figures/simplicial-realization.png",
        "inspection_target": "faces, intersections, barycentric geometry, and the CW cell view",
        "validation": "face closure, intersection property, d1*d2=0, and Betti numbers",
    },
    {
        "item": "applied_cw_diagnostic_lab",
        "concept": "hypothesis checking before applying CW/manifold theorems",
        "representation": "editable table of candidate cell structures",
        "library": "pandas + json",
        "artifact": "tables/cw-diagnostics.csv",
        "inspection_target": "which exact hypothesis licenses or blocks a conclusion",
        "validation": "CSV row count and JSON classification summary",
    },
]

library_rows = [
    {"concept": "convex closed cells", "representation": "sampled radial model", "library": "numpy, matplotlib", "reason": "2D proof geometry and numeric ray checks"},
    {"concept": "attaching maps", "representation": "boundary word on a bouquet", "library": "plotly, sympy", "reason": "inspectable quotient diagram plus exact algebra"},
    {"concept": "CW conditions", "representation": "failure diagrams and dependency graph", "library": "matplotlib, networkx", "reason": "visible limit behavior and theorem dependencies"},
    {"concept": "1-manifold classification", "representation": "regular graph degree signatures", "library": "networkx, matplotlib", "reason": "classification reduces to connected graph patterns"},
    {"concept": "simplicial complexes", "representation": "finite abstract complex and boundary matrices", "library": "gudhi, sympy", "reason": "combinatorial topology and exact chain checks"},
    {"concept": "diagnostic lab", "representation": "editable summary table", "library": "pandas", "reason": "compact comparison of hypotheses and conclusions"},
]

storyboard_path = CHECK_DIR / "visual-storyboard.json"
library_table_path = TABLE_DIR / "library-routing.csv"
save_json(visual_storyboard, storyboard_path)
save_csv(library_rows, library_table_path)

display(Markdown(f"Book root: `{BOOK_ROOT.name}`. Artifact root: `{ARTIFACT_ROOT.relative_to(BOOK_ROOT).as_posix()}`."))
display_artifact(nb_rel(storyboard_path), width=720, height=180)
display_artifact(nb_rel(library_table_path), width=720, height=180)


## 1. Closed Cells as Radial Data

The chapter starts with a useful supply of closed cells: compact convex subsets of Euclidean space with nonempty interior. The proof idea is radial. Pick an interior point, look along every ray starting there, and record the unique point where the ray first reaches the boundary. If each ray has one boundary hit, then the unit ball can be stretched along rays to fill the convex set.

The diagram below uses an ellipse because the formulas are transparent. This is not meant to prove the theorem for every convex body; it is a computational model of the proof move. The invariant to inspect is radial uniqueness. For each direction, the sampled boundary point satisfies the ellipse equation exactly up to floating error. For every shorter radial parameter, the point is strictly inside. The theorem replaces this special formula by compactness, convexity, and the closed map lemma.


In [ ]:
a, b = 1.8, 1.0
angles = np.linspace(0, 2 * np.pi, 361)


def ellipse_radius(theta: np.ndarray | float) -> np.ndarray | float:
    return 1.0 / np.sqrt((np.cos(theta) / a) ** 2 + (np.sin(theta) / b) ** 2)


r = ellipse_radius(angles)
boundary = np.column_stack([r * np.cos(angles), r * np.sin(angles)])
probe_angles = np.deg2rad([15, 55, 105, 160, 225, 300])

fig, ax = plt.subplots(figsize=(7.4, 5.2))
ax.plot(boundary[:, 0], boundary[:, 1], color="#263238", linewidth=2.2, label="boundary of closed cell")
unit = np.column_stack([np.cos(angles), np.sin(angles)])
ax.plot(unit[:, 0], unit[:, 1], color="#9e9e9e", linewidth=1.2, linestyle="--", label="model unit circle")
for theta in probe_angles:
    rb = float(ellipse_radius(theta))
    endpoint = np.array([rb * math.cos(theta), rb * math.sin(theta)])
    ax.plot([0, endpoint[0]], [0, endpoint[1]], color="#1f77b4", alpha=0.75, linewidth=1.5)
    for lam in [0.35, 0.7, 1.0]:
        point = lam * endpoint
        ax.scatter(point[0], point[1], s=22 if lam < 1 else 42, color="#d95f02" if lam == 1 else "#1b9e77", zorder=3)
ax.scatter([0], [0], s=55, color="#111111", zorder=4)
ax.annotate("interior base point", xy=(0, 0), xytext=(-1.55, -0.78), arrowprops={"arrowstyle": "->", "lw": 1})
ax.annotate("one boundary hit per ray", xy=boundary[55], xytext=(0.42, 1.15), arrowprops={"arrowstyle": "->", "lw": 1})
ax.set_title("Radial model for a compact convex closed 2-cell")
ax.set_aspect("equal")
ax.set_xlim(-2.05, 2.05)
ax.set_ylim(-1.35, 1.35)
ax.grid(True, alpha=0.22)
ax.legend(loc="lower right")

boundary_values = (boundary[:, 0] / a) ** 2 + (boundary[:, 1] / b) ** 2
interior_values = []
for theta in np.linspace(0, 2 * np.pi, 97, endpoint=False):
    endpoint = np.array([ellipse_radius(theta) * np.cos(theta), ellipse_radius(theta) * np.sin(theta)])
    for lam in np.linspace(0.05, 0.95, 19):
        point = lam * endpoint
        interior_values.append((point[0] / a) ** 2 + (point[1] / b) ** 2)

radial_check = {
    "model": "ellipse x^2/a^2 + y^2/b^2 <= 1",
    "a": a,
    "b": b,
    "sampled_boundary_count": int(len(boundary_values)),
    "max_boundary_equation_error": float(np.max(np.abs(boundary_values - 1.0))),
    "max_interior_equation_value": float(np.max(interior_values)),
    "all_sampled_interiors_strict": bool(np.max(interior_values) < 1.0),
    "lesson_invariant": "ray parameter lambda < 1 stays in the interior; lambda = 1 is boundary",
}
radial_png = save_matplotlib(fig, FIG_DIR / "radial-cell-homeomorphism.png")
plt.close(fig)
save_json(radial_check, CHECK_DIR / "radial-cell-check.json")

display_artifact(nb_rel(radial_png), width=760)
display_artifact(nb_rel(CHECK_DIR / "radial-cell-check.json"), width=760, height=160)


## 2. Attaching Maps and Characteristic Maps

A cell complex is assembled by maps, not by drawing cells next to each other. To attach an n-cell to a space X, the boundary of a closed n-cell is sent into X by an attaching map. The quotient then identifies each boundary point with its image in X, while the interior remains a new open n-cell.

The next artifact shows a common finite CW pattern: a 0-cell, two 1-cells forming a bouquet, and one 2-cell whose boundary is read as the word `a b a^-1 b^-1`. The exact space is less important here than the bookkeeping. The 1-skeleton has one vertex and two loop edges, so its cycle rank is two. The 2-cell supplies one higher-dimensional cell. The boundary word has zero total exponent in each generator, which is a useful abelianized shadow of an attaching map.

This is the computational version of a characteristic map: we do not need to store every point of the disk. We store the interior cell, the boundary route, and the quotient rule saying where boundary points land.


In [ ]:
t = np.linspace(0, 2 * np.pi, 220)
loop_a_x = -0.82 + 0.82 * np.cos(t)
loop_a_y = 0.58 * np.sin(t)
loop_b_x = 0.82 + 0.82 * np.cos(t)
loop_b_y = 0.58 * np.sin(t)

fig = go.Figure()
fig.add_trace(go.Scatter(x=loop_a_x, y=loop_a_y, mode="lines", name="1-cell a", line={"color": "#1b9e77", "width": 4}))
fig.add_trace(go.Scatter(x=loop_b_x, y=loop_b_y, mode="lines", name="1-cell b", line={"color": "#377eb8", "width": 4}))
fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers+text", text=["0-cell"], textposition="bottom center", name="shared vertex", marker={"color": "#111111", "size": 11}))

square_x = [2.35, 4.05, 4.05, 2.35, 2.35]
square_y = [-0.85, -0.85, 0.85, 0.85, -0.85]
fig.add_trace(go.Scatter(x=square_x, y=square_y, mode="lines", fill="toself", name="2-cell interior", fillcolor="rgba(231,111,81,0.20)", line={"color": "#e76f51", "width": 3}))
edge_labels = [
    (3.20, -1.05, "a"),
    (4.28, 0.00, "b"),
    (3.20, 1.05, "a^-1"),
    (2.12, 0.00, "b^-1"),
]
for x, y, label in edge_labels:
    fig.add_trace(go.Scatter(x=[x], y=[y], mode="text", text=[label], name=f"boundary segment {label}", textfont={"size": 16}))
for x, y in [(2.35, -0.85), (4.05, -0.85), (4.05, 0.85), (2.35, 0.85)]:
    fig.add_trace(go.Scatter(x=[x], y=[y], mode="markers", showlegend=False, marker={"color": "#111111", "size": 7}))
fig.add_annotation(x=-0.1, y=0.95, text="1-skeleton: one vertex, two loops", showarrow=False)
fig.add_annotation(x=3.2, y=1.38, text="attaching map reads the boundary word", showarrow=False)
fig.update_layout(
    title="Attaching a 2-cell by a boundary word",
    width=900,
    height=460,
    xaxis={"visible": False, "scaleanchor": "y", "scaleratio": 1},
    yaxis={"visible": False},
    plot_bgcolor="white",
    margin={"l": 20, "r": 20, "t": 60, "b": 20},
    legend={"orientation": "h", "y": -0.05},
)

attaching_html = save_plotly_html(fig, HTML_DIR / "attaching-map-skeletons.html")
word = ["a", "b", "a^-1", "b^-1"]
d1 = boundary_matrix([(0, 0), (0, 0)], vertex_count=1)
attaching_check = {
    "cell_counts": {"vertices": 1, "edges": 2, "faces": 1},
    "one_skeleton_cycle_rank": cycle_rank_for_graph(vertex_count=1, edge_count=2, component_count=1),
    "euler_characteristic_with_one_2_cell": euler_characteristic(vertices=1, edges=2, faces=1),
    "attaching_word": word,
    "abelianization": abelianization_vector(word, ["a", "b"]),
    "boundary_matrix_d1": [[int(value) for value in row] for row in d1.tolist()],
    "loop_edges_have_zero_vertex_boundary": bool(d1 == sp.zeros(1, 2)),
}
save_json(attaching_check, CHECK_DIR / "attaching-map-check.json")

display_artifact(nb_rel(attaching_html), width=820, height=430)
display_artifact(nb_rel(CHECK_DIR / "attaching-map-check.json"), width=760, height=160)


## 3. Why (C) and (W) Are Not Decorative

The two extra letters in CW are doing real work. Closure finiteness (C) prevents one cell closure from depending on infinitely many lower cells. Coherence (W) prevents a subset from passing every cell-closure test while still failing to be closed globally.

The left panel below models a failure of (W). Each individual cell closure only sees a finite piece of the marked sequence, but the whole space sees the sequence accumulate at the origin. The right panel models a failure of (C). The open disk is one 2-cell, while infinitely many 0-cells and 1-cells are placed around the boundary with an accumulation point. The 2-cell closure touches infinitely many cells, so closure finiteness fails.

The proof dependency graph after the failure modes is a scaffold for several arguments in the chapter. It records which hypotheses power which conclusions. The point is not that this graph proves the theorems, but that it keeps the proof pressure visible: when a conclusion fails, you can look backward and ask whether the missing hypothesis is local finiteness, (C), (W), compactness in a finite subcomplex, or regularity.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
ax = axes[0]
for n in range(1, 16):
    endpoint = np.array([1.0, 1.0 / n])
    ax.plot([0, endpoint[0]], [0, endpoint[1]], color="#90a4ae", linewidth=1)
    if n <= 10:
        point = np.array([1.0 / n, 1.0 / (n * n)])
        ax.scatter(point[0], point[1], color="#d95f02", s=25)
ax.scatter([0], [0], color="#111111", s=45, zorder=4)
ax.scatter([1], [0], color="#546e7a", s=28)
ax.set_title("Failure mode for coherent topology (W)")
ax.set_xlabel("global limit at the origin is missed cell-by-cell")
ax.set_aspect("equal")
ax.set_xlim(-0.05, 1.08)
ax.set_ylim(-0.04, 1.08)
ax.grid(True, alpha=0.2)

ax = axes[1]
theta = np.linspace(0, 2 * np.pi, 400)
ax.fill(np.cos(theta), np.sin(theta), color="#f4a261", alpha=0.18)
ax.plot(np.cos(theta), np.sin(theta), color="#263238", linewidth=1.4)
for n in range(2, 34):
    ang = 2 * np.pi / n
    ax.scatter(np.cos(ang), np.sin(ang), color="#377eb8", s=22)
for n in [2, 3, 4, 5, 8, 13, 21, 33]:
    ang = 2 * np.pi / n
    ax.plot([0, np.cos(ang)], [0, np.sin(ang)], color="#bdbdbd", linewidth=0.7, alpha=0.6)
ax.scatter([1], [0], color="#d95f02", s=44, zorder=4)
ax.set_title("Failure mode for closure finiteness (C)")
ax.set_xlabel("one 2-cell closure touches infinitely many boundary cells")
ax.set_aspect("equal")
ax.set_xlim(-1.12, 1.12)
ax.set_ylim(-1.12, 1.12)
ax.grid(True, alpha=0.2)

failure_png = save_matplotlib(fig, FIG_DIR / "cw-conditions-failure-modes.png")
plt.close(fig)

G = nx.DiGraph()
proof_edges = [
    ("local finiteness", "closure finiteness (C)"),
    ("local finiteness", "coherent topology (W)"),
    ("closure finiteness (C)", "(C) and (W)"),
    ("coherent topology (W)", "(C) and (W)"),
    ("coherent topology (W)", "CW complex connected"),
    ("coherent topology (W)", "coherent skeleta"),
    ("(C) and (W)", "subcomplexes are closed CW complexes"),
    ("(C) and (W)", "maps out checked on cell closures"),
    ("closure finiteness (C)", "compact iff finite subcomplex"),
    ("cell closures compact", "compact iff finite subcomplex"),
    ("1-skeleton connected", "CW complex connected"),
    ("CW complex connected", "1-manifold classification"),
    ("regular 1-CW structure", "degree-two graph pattern"),
    ("degree-two graph pattern", "1-manifold classification"),
    ("inductive attachment", "CW construction theorem"),
    ("coherent skeleta", "inductive attachment"),
]
G.add_edges_from(proof_edges)
fig2, ax2 = plt.subplots(figsize=(9.5, 6.2))
pos = nx.spring_layout(G, seed=12, k=1.05)
nx.draw_networkx_edges(G, pos, ax=ax2, arrows=True, arrowstyle="-|>", arrowsize=14, edge_color="#607d8b", width=1.4)
nx.draw_networkx_nodes(G, pos, ax=ax2, node_size=1550, node_color="#fff3e0", edgecolors="#e76f51", linewidths=1.3)
nx.draw_networkx_labels(G, pos, ax=ax2, font_size=8)
ax2.set_title("Proof and invariant scaffold for Chapter 5")
ax2.axis("off")
proof_png = save_matplotlib(fig2, FIG_DIR / "cw-proof-dependency-graph.png")
plt.close(fig2)

weak_check = {
    "W_failure": {
        "marked_sequence_points_sampled": 10,
        "global_limit_point": [0.0, 0.0],
        "cellwise_closed_but_not_globally_closed": True,
    },
    "C_failure": {
        "sampled_boundary_vertices": 32,
        "single_2_cell_closure_meets_unbounded_number_of_boundary_cells": True,
    },
    "proof_dependency_graph": {
        "nodes": G.number_of_nodes(),
        "edges": G.number_of_edges(),
        "weakly_connected": nx.is_weakly_connected(G),
        "theorem_targets": ["subcomplexes are closed CW complexes", "compact iff finite subcomplex", "1-manifold classification", "CW construction theorem"],
    },
}
save_json(weak_check, CHECK_DIR / "cw-condition-check.json")

display_artifact(nb_rel(failure_png), width=820)
display_artifact(nb_rel(proof_png), width=820)
display_artifact(nb_rel(CHECK_DIR / "cw-condition-check.json"), width=760, height=160)


## 4. Skeletons, Subcomplexes, and What the 1-Skeleton Knows

A skeleton is a controlled truncation. The n-skeleton contains all cells of dimension at most n, and it is a subcomplex. For a CW complex, the topology is coherent with the closed cells, and also coherent with the sequence of skeleta. This is why inductive proofs work: define or check something on one skeleton, extend across the next layer of attached cells, and use coherence to recover a global conclusion.

Connectedness is the best example. Higher-dimensional cells cannot secretly connect two components unless their boundary already lands in the same lower-dimensional side. Thus a CW complex is connected exactly when its 1-skeleton is connected, and connectedness is then path-connectedness. Compactness has a similarly finite form: compact subsets are closed and sit inside finite subcomplexes. The notebook's checks do not prove those theorems in full generality, but they enforce the finite signatures that the theorems use.


## 5. Regular 1-Complexes and the Classification of 1-Manifolds

Once a 1-manifold has a regular CW decomposition, each 1-cell behaves like an interval with two endpoint 0-cells. At an interior manifold point represented by a vertex, exactly two 1-cells must meet. If none meet, the point is isolated. If one meets, the local model is a half-line, which is a boundary model rather than a boundaryless 1-manifold. If three or more meet, deleting the vertex creates too many local components.

So the boundaryless connected case is a degree-two graph. If it closes up after finitely many edges, it is a circle. If the chain never repeats, it is a line. With boundary, degree-one vertices are allowed as boundary points. Two degree-one endpoints give a compact interval. One endpoint with the other direction infinite gives a half-line.

The visual below deliberately shows finite windows for the noncompact models. The JSON check records which endpoints are actual boundary points and which are artificial truncations of an infinite pattern.


In [ ]:
def draw_graph_model(ax, vertices, edges, pos, title, subtitle, vertex_colors=None):
    graph = nx.Graph()
    graph.add_nodes_from(vertices)
    graph.add_edges_from(edges)
    colors = [vertex_colors.get(v, "#111111") if vertex_colors else "#111111" for v in graph.nodes]
    nx.draw_networkx_edges(graph, pos, ax=ax, edge_color="#546e7a", width=2.4)
    nx.draw_networkx_nodes(graph, pos, ax=ax, node_color=colors, node_size=110, edgecolors="white", linewidths=0.8)
    ax.set_title(title)
    ax.text(0.5, -0.12, subtitle, ha="center", va="top", transform=ax.transAxes, fontsize=9)
    ax.axis("off")
    ax.set_aspect("equal")


fig, axes = plt.subplots(2, 2, figsize=(10.4, 7.2))

cycle_vertices = list(range(8))
cycle_edges = [(i, (i + 1) % 8) for i in cycle_vertices]
cycle_pos = {i: (math.cos(2 * math.pi * i / 8), math.sin(2 * math.pi * i / 8)) for i in cycle_vertices}
draw_graph_model(axes[0, 0], cycle_vertices, cycle_edges, cycle_pos, "Compact, no boundary: S^1", "finite connected degree-two pattern")

line_vertices = list(range(-4, 5))
line_edges = [(i, i + 1) for i in range(-4, 4)]
line_pos = {i: (i, 0) for i in line_vertices}
line_colors = {-4: "#d95f02", 4: "#d95f02"}
draw_graph_model(axes[0, 1], line_vertices, line_edges, line_pos, "Noncompact, no boundary: R", "window of a bi-infinite degree-two chain", line_colors)
axes[0, 1].annotate("continues", xy=(-4, 0), xytext=(-5.0, 0.35), arrowprops={"arrowstyle": "->", "lw": 1})
axes[0, 1].annotate("continues", xy=(4, 0), xytext=(4.35, 0.35), arrowprops={"arrowstyle": "->", "lw": 1})

interval_vertices = list(range(6))
interval_edges = [(i, i + 1) for i in range(5)]
interval_pos = {i: (i, 0) for i in interval_vertices}
interval_colors = {0: "#d95f02", 5: "#d95f02"}
draw_graph_model(axes[1, 0], interval_vertices, interval_edges, interval_pos, "Compact with boundary: [0,1]", "two real degree-one endpoints", interval_colors)

ray_vertices = list(range(7))
ray_edges = [(i, i + 1) for i in range(6)]
ray_pos = {i: (i, 0) for i in ray_vertices}
ray_colors = {0: "#d95f02", 6: "#fdae61"}
draw_graph_model(axes[1, 1], ray_vertices, ray_edges, ray_pos, "Noncompact with boundary: [0,infty)", "one real endpoint, one artificial truncation", ray_colors)
axes[1, 1].annotate("continues", xy=(6, 0), xytext=(6.55, 0.35), arrowprops={"arrowstyle": "->", "lw": 1})

fig.tight_layout()
classifier_png = save_matplotlib(fig, FIG_DIR / "one-manifold-classifier.png")
plt.close(fig)

classification_check = {
    "S^1": {"compact": True, "boundary_vertices": 0, "degree_signature": {"all_vertices_degree": 2}, "model": "finite cycle"},
    "R": {"compact": False, "boundary_vertices": 0, "degree_signature": {"all_real_vertices_degree": 2}, "model": "bi-infinite chain"},
    "[0,1]": {"compact": True, "boundary_vertices": 2, "degree_signature": {"two_vertices_degree": 1, "interior_vertices_degree": 2}, "model": "finite path"},
    "[0,infty)": {"compact": False, "boundary_vertices": 1, "degree_signature": {"one_real_vertex_degree": 1, "interior_vertices_degree": 2}, "model": "ray"},
    "finite_cycle_degree_list": sorted(dict(nx.Graph(cycle_edges).degree()).values()),
    "finite_interval_degree_list": sorted(dict(nx.Graph(interval_edges).degree()).values()),
}
save_json(classification_check, CHECK_DIR / "one-manifold-classification.json")

display_artifact(nb_rel(classifier_png), width=820)
display_artifact(nb_rel(CHECK_DIR / "one-manifold-classification.json"), width=760, height=160)


## 6. Simplicial Complexes as Combinatorial CW Complexes

A simplicial complex is a stricter object than a general CW complex. It is made from closed simplices, includes every face of each simplex, and requires intersections to be common faces. Its polyhedron carries a regular CW decomposition by taking the interiors of the simplices as cells.

The finite abstract complex below has two filled triangles, `(0,1,2)` and `(0,2,3)`, plus all of their faces. Geometrically it is a square disk split along a diagonal. The checks do three things. First, they confirm face closure and the intersection property. Second, they build exact boundary matrices and verify `d1*d2 = 0`, the algebraic shadow of "the boundary of a boundary is zero." Third, Gudhi computes Betti numbers, confirming one connected component and no 1-dimensional hole.

This also illustrates why simplicial maps are combinatorial. Once a vertex map sends every simplex's vertex set to a simplex's vertex set in the target, the affine extension on each simplex is forced.


In [ ]:
def all_nonempty_faces(simplex):
    simplex = tuple(simplex)
    faces = []
    for mask in range(1, 1 << len(simplex)):
        faces.append(tuple(simplex[i] for i in range(len(simplex)) if mask & (1 << i)))
    return faces


maximal_simplices = [(0, 1, 2), (0, 2, 3)]
simplices = sorted({tuple(sorted(face)) for simplex in maximal_simplices for face in all_nonempty_faces(simplex)}, key=lambda s: (len(s), s))
vertices = sorted({v for simplex in simplices for v in simplex})
edges = [s for s in simplices if len(s) == 2]
triangles = [s for s in simplices if len(s) == 3]

face_closed = all(tuple(sorted(face)) in simplices for simplex in simplices for face in all_nonempty_faces(simplex))
intersection_ok = True
for i, sigma in enumerate(simplices):
    for tau in simplices[i + 1:]:
        common = tuple(sorted(set(sigma).intersection(tau)))
        if common and (common not in simplices):
            intersection_ok = False

st = gudhi.SimplexTree()
for simplex in maximal_simplices:
    st.insert(simplex, filtration=0.0)
st.persistence()
betti = st.betti_numbers()

edge_index = {edge: i for i, edge in enumerate(edges)}
d1 = boundary_matrix(edges, vertex_count=len(vertices))
d2 = sp.zeros(len(edges), len(triangles))
for j, (a0, a1, a2) in enumerate(triangles):
    oriented_terms = [((a1, a2), 1), ((a0, a2), -1), ((a0, a1), 1)]
    for edge, sign in oriented_terms:
        d2[edge_index[tuple(sorted(edge))], j] = sign
boundary_squared = d1 * d2

coords = {0: np.array([0.0, 0.0]), 1: np.array([1.0, 0.0]), 2: np.array([1.0, 1.0]), 3: np.array([0.0, 1.0])}
fig, ax = plt.subplots(figsize=(6.4, 5.6))
for tri, color in zip(triangles, ["#8ecae6", "#ffb703"]):
    pts = np.array([coords[v] for v in tri] + [coords[tri[0]]])
    ax.fill(pts[:, 0], pts[:, 1], color=color, alpha=0.38)
for edge in edges:
    pts = np.array([coords[v] for v in edge])
    ax.plot(pts[:, 0], pts[:, 1], color="#263238", linewidth=2)
for v in vertices:
    ax.scatter(coords[v][0], coords[v][1], s=80, color="#111111", zorder=4)
    ax.text(coords[v][0] + 0.035, coords[v][1] + 0.035, str(v), fontsize=12)
ax.text(0.52, 0.48, "diagonal face", ha="center", va="center", rotation=45, color="#6d4c41")
ax.set_title("Geometric realization of a finite abstract complex")
ax.set_aspect("equal")
ax.set_xlim(-0.15, 1.18)
ax.set_ylim(-0.15, 1.18)
ax.grid(True, alpha=0.2)

simplicial_png = save_matplotlib(fig, FIG_DIR / "simplicial-realization.png")
plt.close(fig)

simplicial_check = {
    "maximal_simplices": [list(s) for s in maximal_simplices],
    "simplex_count": len(simplices),
    "vertex_count": len(vertices),
    "edge_count": len(edges),
    "triangle_count": len(triangles),
    "face_closed": face_closed,
    "intersection_property_on_vertex_sets": intersection_ok,
    "gudhi_num_simplices": st.num_simplices(),
    "betti_numbers": [int(x) for x in betti],
    "d1_shape": list(d1.shape),
    "d2_shape": list(d2.shape),
    "boundary_squared_zero": bool(boundary_squared == sp.zeros(*boundary_squared.shape)),
    "single_triangle_helper_boundary_squared_zero": simplex_boundary_squared_zero(),
}
save_json(simplicial_check, CHECK_DIR / "simplicial-realization-check.json")

display_artifact(nb_rel(simplicial_png), width=640)
display_artifact(nb_rel(CHECK_DIR / "simplicial-realization-check.json"), width=760, height=160)


## Applied Lab: Diagnose the Hypotheses Before Using the Theorem

The fastest way to misuse a theorem about CW complexes is to identify the cells and forget the hypotheses. This lab stores a few small candidate structures and asks what conclusion is licensed. The columns are intentionally plain: type of object, compactness pattern, closure-finiteness status, coherent-topology status, degree signature when the object is a graph, and the conclusion we are allowed to draw.

To explore, add a row to `lab_cases`, rerun the cell, and compare the status. A connected degree-two finite graph supports the circle model. A finite path supports a compact 1-manifold with boundary. A bouquet with three loops is a CW complex, but it is not a 1-manifold at the wedge point because too many local branches meet. The two pathology rows show why Chapter 5 separates cell decompositions from CW decompositions.


In [ ]:
def degree_summary(vertex_count, edges):
    graph = nx.MultiGraph()
    graph.add_nodes_from(range(vertex_count))
    graph.add_edges_from(edges)
    degrees = dict(graph.degree())
    simple_graph = nx.Graph()
    simple_graph.add_nodes_from(range(vertex_count))
    simple_graph.add_edges_from([edge for edge in edges if edge[0] != edge[1]])
    connected = nx.is_connected(simple_graph) if vertex_count and simple_graph.number_of_edges() else vertex_count == 1
    return {
        "min_degree": min(degrees.values()) if degrees else 0,
        "max_degree": max(degrees.values()) if degrees else 0,
        "degree_multiset": sorted(int(v) for v in degrees.values()),
        "connected": connected,
    }


lab_cases = [
    {"name": "cycle_6", "kind": "finite regular graph", "vertex_count": 6, "edges": [(i, (i + 1) % 6) for i in range(6)], "compact": True, "C": True, "W": True, "actual_boundary_vertices": 0},
    {"name": "interval_5_edges", "kind": "finite regular graph with boundary", "vertex_count": 6, "edges": [(i, i + 1) for i in range(5)], "compact": True, "C": True, "W": True, "actual_boundary_vertices": 2},
    {"name": "bouquet_3_loops", "kind": "finite CW graph", "vertex_count": 1, "edges": [(0, 0), (0, 0), (0, 0)], "compact": True, "C": True, "W": True, "actual_boundary_vertices": 0},
    {"name": "bi_infinite_line_window", "kind": "noncompact graph pattern", "vertex_count": 9, "edges": [(i, i + 1) for i in range(8)], "compact": False, "C": True, "W": True, "actual_boundary_vertices": 0, "artificial_truncations": 2},
    {"name": "ray_window", "kind": "noncompact graph pattern with boundary", "vertex_count": 7, "edges": [(i, i + 1) for i in range(6)], "compact": False, "C": True, "W": True, "actual_boundary_vertices": 1, "artificial_truncations": 1},
    {"name": "comb_like_W_failure", "kind": "cell decomposition pathology", "vertex_count": None, "edges": [], "compact": False, "C": True, "W": False, "actual_boundary_vertices": None},
    {"name": "disk_boundary_C_failure", "kind": "cell decomposition pathology", "vertex_count": None, "edges": [], "compact": True, "C": False, "W": True, "actual_boundary_vertices": None},
]

rows = []
for case in lab_cases:
    row = {key: value for key, value in case.items() if key != "edges"}
    if case["vertex_count"] is not None:
        summary = degree_summary(case["vertex_count"], case["edges"])
        row.update(summary)
        if case["name"] == "cycle_6":
            conclusion = "connected compact boundaryless 1-manifold model: S^1"
        elif case["name"] == "interval_5_edges":
            conclusion = "connected compact 1-manifold with boundary model: [0,1]"
        elif case["name"] == "bouquet_3_loops":
            conclusion = "CW graph but not a 1-manifold at the wedge point"
        elif case["name"] == "bi_infinite_line_window":
            conclusion = "finite window of noncompact boundaryless model: R"
        elif case["name"] == "ray_window":
            conclusion = "finite window of noncompact boundary model: [0,infty)"
        else:
            conclusion = "graph case needs inspection"
    else:
        row.update({"min_degree": None, "max_degree": None, "degree_multiset": None, "connected": None})
        conclusion = "not a CW complex because a required CW condition fails"
    row["licensed_conclusion"] = conclusion
    row["is_CW"] = bool(case["C"] and case["W"])
    rows.append(row)

lab_df = pd.DataFrame(rows)
lab_csv = save_csv(lab_df.to_dict(orient="records"), TABLE_DIR / "cw-diagnostics.csv")
lab_check = {
    "row_count": int(len(lab_df)),
    "cw_rows": int(lab_df["is_CW"].sum()),
    "pathology_rows": int((~lab_df["is_CW"]).sum()),
    "contains_models": ["S^1", "[0,1]", "R", "[0,infty)"],
    "bouquet_rejected_as_manifold": "not a 1-manifold" in lab_df.loc[lab_df["name"] == "bouquet_3_loops", "licensed_conclusion"].iloc[0],
}
save_json(lab_check, CHECK_DIR / "lab-diagnostics.json")

display(lab_df[["name", "kind", "C", "W", "compact", "degree_multiset", "licensed_conclusion"]])
display_artifact(nb_rel(lab_csv), width=760, height=160)
display_artifact(nb_rel(CHECK_DIR / "lab-diagnostics.json"), width=760, height=160)


## Takeaways

A cell decomposition is useful only when the cells control the topology. Characteristic maps tell how each open cell is obtained from a closed model cell, and attaching maps say where its boundary lands in lower dimensions. For finite complexes this is often enough, but infinite complexes need the two CW conditions: closure finiteness and coherent topology.

The skeleton viewpoint is the chapter's main organizing device. Build from the 0-skeleton upward, attach n-cells to the previous skeleton, and check global maps or closed sets by checking the pieces that coherence allows. This is why connectedness is visible in the 1-skeleton and why compactness in a CW complex is equivalent to being closed inside a finite subcomplex.

For 1-manifolds, the topology collapses to a graph classification once a regular CW decomposition is present. Boundaryless connected models are degree-two chains: finite means `S^1`, infinite means `R`. With boundary, degree-one endpoints give `[0,1]` or `[0,infty)` depending on compactness.

Simplicial complexes are a combinatorial subcase of regular CW complexes. Their face and intersection rules make many topological questions finite and checkable, while their polyhedra provide actual spaces rather than mere incidence tables.


In [ ]:
from utils.validation import image_stats  # noqa: E402

required_artifacts = [
    FIG_DIR / "radial-cell-homeomorphism.png",
    HTML_DIR / "attaching-map-skeletons.html",
    FIG_DIR / "cw-conditions-failure-modes.png",
    FIG_DIR / "cw-proof-dependency-graph.png",
    FIG_DIR / "one-manifold-classifier.png",
    FIG_DIR / "simplicial-realization.png",
    CHECK_DIR / "visual-storyboard.json",
    CHECK_DIR / "radial-cell-check.json",
    CHECK_DIR / "attaching-map-check.json",
    CHECK_DIR / "cw-condition-check.json",
    CHECK_DIR / "one-manifold-classification.json",
    CHECK_DIR / "simplicial-realization-check.json",
    CHECK_DIR / "lab-diagnostics.json",
    TABLE_DIR / "library-routing.csv",
    TABLE_DIR / "cw-diagnostics.csv",
]
assert_artifacts(required_artifacts, min_bytes=64)


def load_check(name):
    return json.loads((CHECK_DIR / name).read_text(encoding="utf-8"))


radial = load_check("radial-cell-check.json")
attaching = load_check("attaching-map-check.json")
conditions = load_check("cw-condition-check.json")
classification = load_check("one-manifold-classification.json")
simplicial = load_check("simplicial-realization-check.json")
lab = load_check("lab-diagnostics.json")
storyboard = json.loads((CHECK_DIR / "visual-storyboard.json").read_text(encoding="utf-8"))

assert len(storyboard) >= 7
assert radial["all_sampled_interiors_strict"]
assert radial["max_boundary_equation_error"] < 1e-12
assert attaching["euler_characteristic_with_one_2_cell"] == 0
assert attaching["one_skeleton_cycle_rank"] == 2
assert attaching["abelianization"] == {"a": 0, "b": 0}
assert attaching["loop_edges_have_zero_vertex_boundary"]
assert conditions["W_failure"]["cellwise_closed_but_not_globally_closed"]
assert conditions["C_failure"]["single_2_cell_closure_meets_unbounded_number_of_boundary_cells"]
assert conditions["proof_dependency_graph"]["weakly_connected"]
assert classification["S^1"]["compact"] and classification["S^1"]["boundary_vertices"] == 0
assert classification["[0,1]"]["boundary_vertices"] == 2
assert simplicial["face_closed"] and simplicial["intersection_property_on_vertex_sets"]
assert simplicial["boundary_squared_zero"] and simplicial["betti_numbers"][:2] == [1, 0]
assert lab["row_count"] >= 7 and lab["bouquet_rejected_as_manifold"]

png_stats = [image_stats(path) for path in required_artifacts if path.suffix == ".png"]
assert png_stats and all(item["max_channel_stddev"] > 1.0 for item in png_stats)

final_sanity = {
    "artifact_count": len(required_artifacts),
    "png_count": len(png_stats),
    "all_artifacts_nonempty": True,
    "radial_max_boundary_error": radial["max_boundary_equation_error"],
    "attaching_euler_characteristic": attaching["euler_characteristic_with_one_2_cell"],
    "simplicial_betti_numbers": simplicial["betti_numbers"],
    "lab_rows": lab["row_count"],
    "nonblank_pngs": [item["path"] for item in png_stats],
}
final_path = save_json(final_sanity, CHECK_DIR / "final-sanity.json")
assert_artifacts([final_path], min_bytes=64)
final_sanity
